# 09. К6: дообучение декодера VAE

Диагностика показала, что VAE сам теряет высокие частоты: без UNet эффективное разрешение 22 см при 8 см у
эталона. Здесь дообучается только декодер VAE на ваших эталонах; энкодер заморожен, поэтому латентное
пространство не меняется и готовые чекпоинты PiSA-SR работают с новым декодером без переобучения.

Порядок:

1. Дообучить декодер (E16–E18) и посмотреть, сдвинулся ли потолок VAE.
2. Подставить новый декодер в уже обученную модель и сравнить с исходным декодером — без обучения UNet.
3. Если шаг 2 дал выигрыш, обучить UNet с новым декодером (те же эксперименты E00–E15 с переменной `VAE_DEC`).

LQ и UNet на шаге 1 не нужны. Запускайте варианты E16–E18 по одному; время одного запуска на T4 не замерялось.

## Инструкция

1. **Настройки ноутбука (панель справа).** Accelerator — `GPU T4 x2`, Internet — `On`, Persistence — `Files only`.
2. **Подключите данные (Add Input):**
   - датасет с кодом — папка `PiSA-SR-main` целиком, вместе с `freq_exp`;
   - обучающие пары LQ/HQ и валидационные пары LQ/HQ;
   - Stable Diffusion 2.1-base, релизный чекпоинт `pisa_sr.pkl` и, если используете теги, `ram_swin_large_14m.pth`.
3. **Ячейка «Пути».** Впишите свои пути. Ячейка «Проверка» сразу скажет, чего не хватает.
4. **Запускайте ячейки по порядку.** Обучение печатает строку каждые 20 шагов и валидацию каждые `EVAL_EVERY` шагов (по умолчанию 500).
5. **Время.** Сессия Kaggle ограничена; если этап не успевает, уменьшите `STEPS` или запускайте этапы по отдельности
   (см. раздел «Если сессия оборвалась» в конце).
6. **Сохраните результат.** Последняя ячейка упаковывает логи и чекпоинты в архив. Нажмите `Save Version` →
   `Save & Run All` либо скачайте архив из `/kaggle/working`. Для сравнения экспериментов
   (ноутбук `08_compare_all`) подключите сохранённые результаты как входные данные.

**Что смотреть в результатах**

- `curve.png` и `curve_*.csv` — LPIPS, VoL/GT и ошибка в высокочастотных полосах по шагам;
- `hf_err` — главная метрика: 1 = детали не восстановлены, больше 1 = выдуманы, меньше — лучше;
- `eff_gsd_cm` — эффективное разрешение в сантиметрах (меньше — лучше);
- `train_log_*.csv` — значение, вес и норма градиента каждого члена потерь. Если норма градиента частотного
  члена меньше 5 % от главного (L2 в pix-этапе, LPIPS в sem-этапе), член почти не влияет — увеличьте вес.

In [ ]:
NOTEBOOK_TAG = "09_K6_vae"

# ==== Пути: отредактируйте под свои датасеты ====
CODE_SRC  = "/kaggle/input/pisa-sr-main/PiSA-SR-main"      # папка с кодом (внутри должна быть freq_exp)
SD_PATH   = "/kaggle/working/sd_v2_1_full"                  # Stable Diffusion 2.1-base
INIT_CKPT = "/kaggle/input/pisa-sr-weights/pisa_sr.pkl"     # релизный чекпоинт PiSA-SR
RAM_PATH  = ""                                              # ram_swin_large_14m.pth; пусто = без RAM
TRAIN_LQ  = "/kaggle/input/hq-lq-netherlands-train/lq_images"
TRAIN_HQ  = "/kaggle/input/hq-lq-netherlands-train/hq_images"
VAL_LQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/lq_images"
VAL_HQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/hq_images"

# ==== Параметры обучения ====
OUT_ROOT = "/kaggle/working/experiments/freq"   # сюда пишутся логи и чекпоинты
STEPS    = 4000    # шагов на этап (как в ваших прежних запусках)
EVAL_EVERY = 500   # валидация каждые столько шагов; чаще — точнее кривая, но дольше обучение
ACCUM    = 1       # накопление градиента: 4 сглаживает кривые, но за те же шаги даёт вчетверо меньше обновлений
EMA      = 0       # 0.999 включает EMA весов LoRA (сохраняется отдельный *_ema.pkl)
FDL_CROP = 256     # окно для FDL: 256 экономит память; 0 = весь кроп 512
AUTO     = 0.3     # калибровка весов частотных членов по норме градиента; 0 = взять веса как есть

# ==== К6 ====
VAE_STEPS = 4000      # шагов дообучения декодера
# Чекпоинт PiSA-SR для шага 2: релизный или ваш дообученный (например, pisasr_final.pkl из 01_baseline)
CKPT_TO_CHECK = INIT_CKPT

In [ ]:
import os, shutil, subprocess, sys

# --- Проверка ---
!nvidia-smi --query-gpu=name,memory.total --format=csv
missing = [p for p in [CODE_SRC, SD_PATH, INIT_CKPT, TRAIN_LQ, TRAIN_HQ, VAL_LQ, VAL_HQ] if not os.path.exists(p)]
assert not missing, f"Не найдены пути: {missing}"
assert os.path.isdir(os.path.join(CODE_SRC, "freq_exp")), "В датасете с кодом нет папки freq_exp"
if RAM_PATH and not os.path.exists(RAM_PATH):
    raise FileNotFoundError(RAM_PATH)

# --- Код копируется в рабочую папку (входные датасеты доступны только для чтения) ---
WORK = "/kaggle/working/PiSA-SR-main"
if not os.path.exists(WORK):
    shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)

# --- Пакеты: две функции потерь из статей + метрики ---
!pip install -q focal-frequency-loss fdl-pytorch lpips pyiqa
# Если окружение чистое, раскомментируйте установку зависимостей PiSA-SR:
# !pip install -q -r requirements.txt

os.environ.update(SD=SD_PATH, INIT=INIT_CKPT, RAM=RAM_PATH, TRAIN_LQ=TRAIN_LQ, TRAIN_HQ=TRAIN_HQ,
                  VAL_LQ=VAL_LQ, VAL_HQ=VAL_HQ, OUT=OUT_ROOT, STEPS=str(STEPS), ACCUM=str(ACCUM),
                  EMA=str(EMA), FDL_CROP=str(FDL_CROP), AUTO=str(AUTO), EVAL_EVERY=str(EVAL_EVERY))
print("Рабочая папка:", os.getcwd())
!bash freq_exp/experiments.sh list

os.environ["VAE_STEPS"] = str(VAE_STEPS)

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display

def show_vae(exp_prefix):
    """Кривые дообучения декодера: первая строка — исходный потолок VAE, последняя — новый."""
    d = sorted(glob.glob(os.path.join(OUT_ROOT, exp_prefix + "*")))[0]; print("==", d)
    if os.path.exists(os.path.join(d, "curve_vae.png")):
        display(Image(os.path.join(d, "curve_vae.png"), width=700))
    c = pd.read_csv(os.path.join(d, "curve_vae.csv")); display(pd.concat([c.head(1), c.tail(3)]))
    b = pd.read_csv(os.path.join(d, "bands_vae.csv"))
    print("По полосам (доля Найквиста, b0 = 0–0,125 ... b7 = 0,875–1): шаг 0 и последний шаг")
    display(b[b.global_step.isin([0, b.global_step.max()])])
    t = pd.read_csv(os.path.join(d, "train_log_vae.csv")).dropna(axis=1, how="all")
    print("Веса и нормы градиентов членов потерь (последние записи):"); display(t.tail(3))
    return os.path.join(d, "checkpoints", "vae_decoder_final.pt")

## Шаг 1. Дообучение декодера

- **E16** — контроль: только L1 + LPIPS. Показывает, что даёт сама подстройка под аэроснимки.
- **E17** — L1 + LPIPS + Focal Frequency Loss.
- **E18** — то же и FDL.

Первая строка `[eval] шаг 0` — исходный потолок VAE (должна совпасть с пунктом 2 из `00_diagnostics`). В конце печатается строка «Было -> стало».

In [ ]:
!bash freq_exp/experiments.sh E16

In [ ]:
dec_E16 = show_vae("E16")

In [ ]:
!bash freq_exp/experiments.sh E17

In [ ]:
dec_E17 = show_vae("E17")

In [ ]:
!bash freq_exp/experiments.sh E18

In [ ]:
dec_E18 = show_vae("E18")

## Шаг 2. Новый декодер в готовой модели, без обучения UNet

Один и тот же чекпоинт PiSA-SR прогоняется с исходным и с новым декодером. Это проверка главного риска: декодер учился на латентах эталонов, а при сверхразрешении получает латенты от UNet.

Впишите в `VAE_DEC` лучший декодер из шага 1.

In [ ]:
VAE_DEC = dec_E17
N_EVAL = 100
!python freq_exp/infer_eval.py --pretrained_model_path {SD_PATH} --ckpt {CKPT_TO_CHECK} \
    --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --out_dir /kaggle/working/sr/K6_old_decoder --n {N_EVAL}
!python freq_exp/infer_eval.py --pretrained_model_path {SD_PATH} --ckpt {CKPT_TO_CHECK} \
    --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --out_dir /kaggle/working/sr/K6_new_decoder --n {N_EVAL} \
    --vae_decoder_ckpt {VAE_DEC}

In [ ]:
!python freq_exp/eval_bands.py --sr_dir /kaggle/working/sr/K6_new_decoder/lsem_1.0 \
    --sr_dir_b /kaggle/working/sr/K6_old_decoder/lsem_1.0 --hq_dir {VAL_HQ}

## Шаг 3. Обучение UNet с новым декодером

Любой эксперимент E00–E15 запускается с переменной `VAE_DEC`; результат пишется в папку с окончанием `_vaedec`, прежние запуски не затираются. Сравнивать нужно с тем же экспериментом без `VAE_DEC`.

К6 из отчёта — это К2 (E09) с новым декодером. Оба этапа E09 в одну сессию могут не поместиться.

In [ ]:
os.environ["VAE_DEC"] = VAE_DEC
!bash freq_exp/experiments.sh E09

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display

def show(exp_prefix):
    """Кривые, последние точки валидации и нормы градиентов для эксперимента."""
    dirs = sorted(glob.glob(os.path.join(OUT_ROOT, exp_prefix + "*")))
    assert dirs, f"нет результатов для {exp_prefix}"
    d = dirs[0]; print("==", d)
    if os.path.exists(os.path.join(d, "curve.png")):
        display(Image(os.path.join(d, "curve.png"), width=700))
    cols = ["global_step", "stage", "lpips_val", "vol_ratio", "hf_frc", "hf_err", "hf_energy", "eff_gsd_cm", "mem0_gb", "mem1_gb"]
    for f in sorted(glob.glob(os.path.join(d, "curve_*.csv"))):
        c = pd.read_csv(f); display(c[cols].tail(5))
    for f in sorted(glob.glob(os.path.join(d, "train_log_*.csv"))):
        t = pd.read_csv(f).dropna(axis=1, how="all")
        g = t[[c for c in t.columns if c.startswith(("global_step", "stage", "w_", "gnorm_"))]]
        print("Веса и нормы градиентов членов потерь (последние записи):"); display(g.tail(4))

show("E09_K2_daw_fdl_regions_vaedec")

In [ ]:
# Архив с логами и итоговыми чекпоинтами (промежуточные чекпоинты не включаются).
import zipfile
zpath = "/kaggle/working/results_" + NOTEBOOK_TAG + ".zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_ROOT):
        for fn in files:
            if fn.endswith((".csv", ".png", ".json")) or fn.endswith(("final.pkl", "final.pt")):
                p = os.path.join(root, fn); z.write(p, os.path.relpath(p, os.path.dirname(OUT_ROOT)))
    if os.path.isdir("/kaggle/working/sr"):
        for root, _, files in os.walk("/kaggle/working/sr"):
            for fn in files:
                if fn.endswith(".csv"):
                    p = os.path.join(root, fn); z.write(p, os.path.relpath(p, "/kaggle/working"))
print(zpath, round(os.path.getsize(zpath) / 2**20, 1), "МБ")

## Как читать результат

- **Потолок не сдвинулся (шаг 1).** Потери — в самом латенте с 8-кратным сжатием, декодером их не вернуть.
  Это отрицательный результат, его стоит привести в дипломе как обоснование ограничения метода.
- **Потолок вырос, а на шаге 2 выигрыша нет или стало хуже.** Декодер подстроился под латенты эталонов.
  Тогда судить по шагу 3: там UNet обучается уже с новым декодером.
- **Энергия ВЧ выросла, а ошибка в ВЧ не упала.** Декодер рисует текстуру, но не на месте: резкость выше,
  точность та же.

## Если что-то пошло не так

- **Не хватает памяти.** Добавьте к команде `--batch_size 2`; кроп 256 не увеличивайте.
- **Потеря стала NaN.** Уменьшите `--lr` до 5e-6 и оставьте `AUTO = 0.3`.
- **Сессия оборвалась.** Каждые 200 шагов сохраняется `checkpoints/vae_decoder_last.pt`; продолжить можно
  той же командой с `--init_decoder <путь>` и меньшим `--steps` (счёт шагов начнётся заново).